In [4]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [7]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [8]:
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from src.preprocess import preprocess_data

In [9]:
print("Loading and preprocessing training data...")
train_data = pd.read_csv(r'../data/raw/train.csv')
df = preprocess_data(train_data)

X = df.drop('health_condition', axis=1)
y = df['health_condition']

Loading and preprocessing training data...


In [10]:
print("Starting Optuna Hyperparameter Tuning with Sample Weights...")

X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
def objective(trial):
    # 2. Define the hyperparameter search space
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }

    model = XGBClassifier(**param)
    
    # 4. Calculate sample weights for the local training data ONLY
    train_weights = compute_sample_weight(
        class_weight='balanced',
        y=y_train_local
    )

    model.fit(X_train_local, y_train_local, sample_weight=train_weights)
    preds = model.predict(X_test_local)
    macro_f1 = f1_score(y_test_local, preds, average='macro')
    return macro_f1
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\n--- Tuning Complete ---")
print(f"Best Macro F1-Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

Starting Optuna Hyperparameter Tuning with Sample Weights...


[I 2026-07-29 21:33:39,691] A new study created in memory with name: no-name-15b56245-1e47-43d4-8b59-8166f00c08d4
[I 2026-07-29 21:33:51,734] Trial 0 finished with value: 0.7717673474184167 and parameters: {'n_estimators': 311, 'learning_rate': 0.16004997428355586, 'max_depth': 4, 'subsample': 0.9556708662680662, 'colsample_bytree': 0.8114822411318474, 'min_child_weight': 1}. Best is trial 0 with value: 0.7717673474184167.
[I 2026-07-29 21:34:04,815] Trial 1 finished with value: 0.7531870268995804 and parameters: {'n_estimators': 276, 'learning_rate': 0.032433165064385906, 'max_depth': 6, 'subsample': 0.9821252976843178, 'colsample_bytree': 0.5351161862555176, 'min_child_weight': 6}. Best is trial 0 with value: 0.7717673474184167.
[I 2026-07-29 21:34:19,563] Trial 2 finished with value: 0.7366509938827517 and parameters: {'n_estimators': 420, 'learning_rate': 0.029411367369653146, 'max_depth': 3, 'subsample': 0.85683894618236, 'colsample_bytree': 0.5925307700260394, 'min_child_weight':


--- Tuning Complete ---
Best Macro F1-Score: 0.8868
Best Hyperparameters:
    n_estimators: 485
    learning_rate: 0.19233650728925847
    max_depth: 10
    subsample: 0.6767655133451045
    colsample_bytree: 0.9786605812318665
    min_child_weight: 6


In [12]:
print("Retraining final model on ALL data with Sample Weights...")

# 1. Calculate weights for the entire 100% dataset
full_weights = compute_sample_weight(
    class_weight='balanced',
    y=y
)

# 2. Build the final model unpacking the best Optuna parameters
final_model = XGBClassifier(
    **study.best_params, 
    random_state=42, 
    n_jobs=-1, 
    eval_metric='mlogloss'
)

final_model.fit(X, y, sample_weight=full_weights)

print("Model successfully trained! Ready to process test.csv...")

Retraining final model on ALL data with Sample Weights...
Model successfully trained! Ready to process test.csv...


In [13]:
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = final_model.predict(X_test_kaggle)

# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_7.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
